In [ ]:
import re
import warnings
def get_reflectance_bidirectional(macket_scn,phase_scn):
    try :
        pattern_id = re.compile(r"scene\.objects\.object[\d_\-]+\.dartNameId=fg0")
        with open(macket_scn, "r", encoding="utf-8") as file:
            lines = file.readlines()
        for i, line in enumerate(lines):
            if pattern_id.search(line):
                break
        target_line = lines[i + 4].strip()
        value = target_line.split('=', 1)[1]
        pattern_ref = re.compile(fr"scene\.materials\.{re.escape(value)}\.ref=")
        for i, line in enumerate(lines):
            if pattern_ref.search(line):
                break
        ref = line.split('=', 1)[1]
        ref = ref.strip()
        pattern_value = re.compile(fr"scene\.materials\.{re.escape(ref)}\.(kd|kr|r0)=(.+)")
        with open(phase_scn, "r", encoding="utf-8") as file:
            lines = file.readlines()
            reflectance_values=None
        for i, line in enumerate(lines):
            if pattern_value.search(line):
                values = line.split('=', 1)[1]
                reflectance_values=values.split(' ')
                reflectance_values=[float(i.strip()) for i in reflectance_values]
                
            if reflectance_values is None :
                pattern_value = re.compile(r"scene\.materials\.lambertian0\.(kd|kr|r0)=(.+)")
                for i, line in enumerate(lines):
                    if pattern_value.search(line):
                        values = line.split('=', 1)[1]
                        reflectance_values=values.split(' ')
                        reflectance_values=[float(i.strip()) for i in reflectance_values]
                        warnings.warn("-------- This optical property can not be associated to the ground with AI model, the first Lambertian optical property is associated to the ground,    -----------")
        return reflectance_values

    except :
        raise ValueError("--------- Scene Reflectance Value Not Found. ---------") 

In [16]:
get_reflectance_bidirectional('/home/fayari/DART_1434/user_data/simulations/test_AI/output/maket.scn','/home/fayari/DART_1434/user_data/simulations/test_AI/output/phase.scn')

/tmp/ipykernel_1649748/2549301119.py:36: UserWarning: -------- Hapke optical property can not be associated to the ground with AI model, Lambertian optical property is associated to the ground,    -----------
  warnings.warn("-------- Hapke optical property can not be associated to the ground with AI model, Lambertian optical property is associated to the ground,    -----------")


[0.042440964852, 0.042440964852, 0.042440964852]